# Loop 4 · Autonomous: bad escalation on new traffic

The growth team added a retention playbook to AGENTS.md (20% off for anyone who mentions
cancelling, release quarters for upcoming features) and shipped it without the gate. Then a new
kind of question starts arriving: discounts, roadmap dates, contracts.
`knowledge/support_process.md` says escalate.

| Loop | Failure | Signal | Store | Update | Promotes |
|---|---|---|---|---|---|
| 1 | Skips the lookup | you | you | you | you |
| 2 | Makes things up | you + a grounding judge | triage + approve | the analyst | you |
| 3 | Unsafe account help, from another team's edit | you + an aligned judge, must-pass check | triage + approve | an optimizer | you |
| 4 | Bad escalation, from another team's edit | a 50% label sample + scheduled scoring | `cycle` | `cycle` | the gate |

In this loop nobody runs the steps. Reviewers label **half** of the new traffic; `selfimprove cycle`
does the rest in one MLflow run and, with `SI_AUTO_PROMOTE=true`, the gate promotes. `monitor`
then watches live traffic and rolls back if the new version does worse than the one it replaced.

In [ ]:
import _session as s

experiment_id = s.setup(4)
r = s.results(4)

In [ ]:
from selfimprove import instructions

c = r["outside_change"]
print(f"v{c['version']} by {c['by']}: {c['message']}
")
print(instructions.diff(c["base_version"], c["version"]))

## 1. Find: new traffic

In [ ]:
before = s.loop_traces(4, "before")
s.show(s.answers(before))

## 2-5. One `cycle`

`cycle` = score → monitor → align → triage → approve → (enough failures?) → improve → gate. Its
guardrails are what make it safe unattended: no signal, no change; a judge that disagrees with
people halts the cycle; the proposer never approves; the holdout stays hidden; must-pass, margin
and bloat rules at the gate; rollback after.

In [ ]:
print(r["cycle"]["outcome"], "-", r["cycle"]["message"])
r["cycle"]["metrics"]

In [ ]:
import mlflow

cycle_run = mlflow.search_runs(filter_string="tags.`si.job` = 'cycle'", order_by=["start_time DESC"],
                               output_format="list")[0]
steps = mlflow.search_runs(filter_string=f"tags.mlflow.parentRunId = '{cycle_run.info.run_id}'",
                           order_by=["start_time ASC"], output_format="list")
s.show(s.pd.DataFrame([{"job": x.data.tags.get("si.job"), "outcome": x.data.tags.get("si.outcome"),
                        "message": (x.data.tags.get("si.message") or "")[:110]} for x in steps]))

In [ ]:
print(instructions.diff(r["production_before"], r["candidate"]))

In [ ]:
s.show(s.gate_table(4), pct=True)

## 6. Measure, and monitor

In [ ]:
s.show(s.before_after(4))

In [ ]:
print(r["monitor"]["outcome"], "-", r["monitor"]["message"])

## The whole run

Every promotion is an alias move with a record: who promoted, from which gate run, and the live
failure rate it replaced.

In [ ]:
from mlflow import MlflowClient
from selfimprove import config

rows = []
for v in sorted(MlflowClient().search_prompt_versions(config.settings.prompt_name).prompt_versions,
                key=lambda p: int(p.version)):
    t = v.tags or {}
    rows.append({"version": v.version, "author": t.get("author", ""), "gate": t.get("gate", ""),
                 "promoted_by": t.get("promoted_by", ""), "message": (v.commit_message or "")[:90]})
s.show(s.pd.DataFrame(rows))

In [ ]:
s.show(s.pd.DataFrame({f"loop {n}": {"failure before": s.results(n)["before"]["failure_rate"],
                                    "failure after": s.results(n).get("after", {}).get("failure_rate")}
                       for n in (1, 2, 3, 4)}).T, pct=True)

**What stays with a person**, even here: the golden cases, labels on a sample of traffic, the
human queue (contradictions, flagged tests, halts), the must-pass rules and the promotion
margin. `.github/workflows/selfimprove-loop.yml` runs the same jobs on a schedule.